# pipelines.decider

> Typed, calibrated decisions via the shared `/v1/systemone` (System One / Jev) API.

A **System One** decision model reads a piece of application *state* plus one or
more typed *questions* and returns typed, calibrated *answers* (a `choice`, a
`score`, or a `noul`) in a single request. The same `/v1/systemone` HTTP contract
is implemented by multiple providers (TypeSafe/Jev, Ollama 0.35+, and other
compatible servers), so one thin client works across all of them — you only
change the `base_url`, `api_key`, and `model`.

```python
from onprem.pipelines import Decider

# Ollama (local)
d = Decider(base_url="http://localhost:11434", api_key="ollama", model="nimble")

# TypeSafe / Jev cloud (reads TYPESAFE_API_KEY from the environment)
d = Decider(base_url="https://api.typesafe.ai", model="jev-latest")

resp = d.ask(
    "I was charged twice. Please refund the extra payment.",
    choices=[("Which team should handle this?", ["billing", "technical", "other"])],
    nouls=["Does the customer explicitly ask for a refund?"],
    scores=[("How urgent is this ticket?", ["Routine", "Soon", "Urgent"])],
)
```

In [ ]:
#| default_exp pipelines.decider


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export

import json
import os
import random
import time
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union


# Endpoint paths appended to `base_url` for each supported protocol.
_SYSTEMONE_PATH = "/v1/systemone"   # legacy TypeSafe/Ollama "System One" format
_DECISIONS_PATH = "/v1/decisions"   # OpenAI "Decisions" format (gpt-6-luna, etc.)
_PROTOCOL_PATHS = {"systemone": _SYSTEMONE_PATH, "decisions": _DECISIONS_PATH}

# Statuses that should NOT be retried (client errors), per the API spec.
_NO_RETRY_STATUSES = {400, 401, 422}
# Statuses that SHOULD be retried with exponential backoff, per the API spec.
_RETRY_STATUSES = {429, 500, 502, 503, 504, 529}


class SystemOneError(Exception):
    """Raised when the System One endpoint returns an error response."""

    def __init__(self, message: str, status_code: Optional[int] = None):
        super().__init__(message)
        self.status_code = status_code


class _Answer:
    """Lightweight, attribute-accessible view over one answer in the response.

    Mirrors the fields of the `/v1/systemone` response so callers can use
    `answer.choice`, `answer.noul`, `answer.score`, `answer.probabilities`, and
    `answer.confidence` regardless of question type (absent fields are `None`).

    When the response comes from the OpenAI `/v1/decisions` protocol, the fields
    are normalized into this same shape: `predicate.probability` becomes `noul`,
    and the `probabilities` list `[{value, probability}, ...]` becomes the
    `{value: probability}` map used by System One.
    """

    __slots__ = ("type", "choice", "noul", "score", "legend", "probabilities",
                 "confidence", "raw")

    def __init__(self, data: Dict[str, Any], protocol: str = "systemone"):
        self.raw = data
        self.type = data.get("type")
        self.legend = data.get("legend")
        self.confidence = data.get("confidence")
        self.score = data.get("score")

        if protocol == "decisions":
            # predicate -> noul; keep choice/score as-is.
            self.noul = data.get("probability")
            self.choice = data.get("choice")
            # probabilities: list[{value, probability}] -> {value: probability}
            probs = data.get("probabilities")
            if isinstance(probs, list):
                self.probabilities = {
                    p.get("value"): p.get("probability") for p in probs
                }
            else:
                self.probabilities = probs
        else:
            self.choice = data.get("choice")
            self.noul = data.get("noul")
            self.probabilities = data.get("probabilities")

    def __repr__(self):
        return f"_Answer({self.raw!r})"


class SystemOneResponse:
    """Parsed response from a decision endpoint (`/v1/systemone` or `/v1/decisions`).

    **Attributes:**

    - *model*: The model that produced the answers.
    - *answers*: A dict mapping each question key/name to an `_Answer`.
    - *usage*: Token-usage dict (the billing basis).
    - *raw*: The raw decoded JSON response.

    The `/v1/decisions` response returns `answers` as a list of objects each
    carrying a `name`; this is normalized to the same `{name: _Answer}` map used
    by System One so downstream code is identical across protocols.
    """

    def __init__(self, data: Dict[str, Any], protocol: str = "systemone"):
        self.raw = data
        self.model = data.get("model")
        self.usage = data.get("usage")
        raw_answers = data.get("answers")
        answers: Dict[str, _Answer] = {}
        if protocol == "decisions":
            # answers is a list; key each by its `name`.
            for item in (raw_answers or []):
                answers[item.get("name")] = _Answer(item, protocol="decisions")
        else:
            for key, val in (raw_answers or {}).items():
                answers[key] = _Answer(val, protocol="systemone")
        self.answers: Dict[str, _Answer] = answers

    def __repr__(self):
        return f"SystemOneResponse(model={self.model!r}, answers={list(self.answers)})"


class Decider:
    def __init__(
        self,
        base_url: Optional[str] = None, # Base URL of a decision server (e.g., "http://localhost:11434" for Ollama, "https://api.typesafe.ai" for Jev, "https://api.openai.com" for OpenAI). Falls back to the `TYPESAFE_BASE_URL` environment variable.
        model: Optional[str] = None, # Model name/route (e.g., "nimble", "jev-latest", "gpt-6-luna"). Falls back to the `TYPESAFE_DEFAULT_MODEL` environment variable.
        api_key: Optional[str] = None, # Bearer API key. Falls back to the `TYPESAFE_API_KEY` environment variable. Use "ollama" (or any placeholder) for a local Ollama server.
        protocol: str = "systemone", # Wire protocol: "systemone" (legacy TypeSafe/Ollama `/v1/systemone`, the default) or "decisions" (OpenAI `/v1/decisions`). Falls back to the `DECIDER_PROTOCOL` environment variable.
        timeout: int = 120, # Per-request timeout in seconds.
        max_retries: int = 3, # Number of retries for retryable statuses (429/5xx) using exponential backoff with jitter.
        verify: Optional[Union[bool, str]] = None, # TLS verification passed to `requests`. A path to a custom CA bundle (e.g., a corporate root CA behind a TLS-intercepting firewall), or `True`/`False`. When `None` (default), verification is left to `requests` (which honors the `REQUESTS_CA_BUNDLE`/`CURL_CA_BUNDLE` env vars, then `certifi`). Also falls back to the `TYPESAFE_CA_BUNDLE` environment variable.
        cert: Optional[Union[str, Tuple[str, str]]] = None, # Optional client-side certificate for mutual TLS, passed to `requests` as `cert`: either a path to a combined cert+key PEM, or a `(cert_path, key_path)` tuple. Required by some corporate proxies.
        **kwargs, # Reserved for future options.
    ):
        """
        `Decider` sends application *state* plus one or more typed *questions* to
        a decision model and returns typed, calibrated *answers*.

        It speaks two interchangeable HTTP contracts, selected via `protocol`:

        - `"systemone"` (default): the legacy TypeSafe/Ollama `/v1/systemone`
          format. Use this for Ollama (which is `/v1/systemone`-only) and
          existing TypeSafe/Jev deployments.
        - `"decisions"`: OpenAI's `/v1/decisions` format (e.g., `gpt-6-luna`).

        Requests are built for the chosen protocol and responses are normalized
        to the same `SystemOneResponse`/`_Answer` shape, so downstream code
        (`.noul`, `.choice`, `.score`, `.probabilities`) is identical either way.
        """
        self.base_url = (base_url or os.environ.get("TYPESAFE_BASE_URL") or "").rstrip("/")
        if not self.base_url:
            raise ValueError(
                "A `base_url` is required (or set the TYPESAFE_BASE_URL environment "
                'variable). Examples: "http://localhost:11434" (Ollama), '
                '"https://api.typesafe.ai" (TypeSafe/Jev), or '
                '"https://api.openai.com" (OpenAI).'
            )
        self.model = model or os.environ.get("TYPESAFE_DEFAULT_MODEL")
        if not self.model:
            raise ValueError(
                "A `model` is required (or set the TYPESAFE_DEFAULT_MODEL environment "
                'variable). Examples: "nimble" (Ollama), "jev-latest" (TypeSafe/Jev), '
                'or "gpt-6-luna" (OpenAI).'
            )
        self.protocol = (protocol or os.environ.get("DECIDER_PROTOCOL") or "systemone").lower()
        if self.protocol not in _PROTOCOL_PATHS:
            raise ValueError(
                f"Unsupported protocol {self.protocol!r}. "
                f"Choose one of: {', '.join(sorted(_PROTOCOL_PATHS))}."
            )
        self.api_key = api_key or os.environ.get("TYPESAFE_API_KEY")
        self.timeout = timeout
        self.max_retries = max_retries
        # TLS options for corporate firewalls / TLS interception. Leaving `verify`
        # as `None` preserves `requests`' default resolution (which honors the
        # REQUESTS_CA_BUNDLE/CURL_CA_BUNDLE env vars, then certifi); only when it
        # is explicitly set (here or via TYPESAFE_CA_BUNDLE) is it forwarded.
        self.verify = verify if verify is not None else os.environ.get("TYPESAFE_CA_BUNDLE")
        self.cert = cert
        self.endpoint = self.base_url + _PROTOCOL_PATHS[self.protocol]

    # ---------------------------------------------------------------- HTTP

    def _post(self, payload: Dict[str, Any]) -> Dict[str, Any]:
        """POST `payload` to the System One endpoint with retry/backoff."""
        import requests  # base dependency; imported lazily to keep import light

        headers = {"Content-Type": "application/json"}
        if self.api_key:
            headers["Authorization"] = f"Bearer {self.api_key}"

        last_exc: Optional[Exception] = None
        for attempt in range(self.max_retries + 1):
            try:
                request_kwargs: Dict[str, Any] = {}
                if self.verify is not None:
                    # Only pass `verify` when explicitly configured so that the
                    # default path keeps honoring REQUESTS_CA_BUNDLE/certifi.
                    request_kwargs["verify"] = self.verify
                if self.cert is not None:
                    request_kwargs["cert"] = self.cert
                resp = requests.post(
                    self.endpoint,
                    data=json.dumps(payload),
                    headers=headers,
                    timeout=self.timeout,
                    **request_kwargs,
                )
            except requests.RequestException as e:
                # Network-level error: retry with backoff, then re-raise.
                last_exc = e
                if attempt < self.max_retries:
                    self._sleep_backoff(attempt)
                    continue
                raise SystemOneError(f"Request to {self.endpoint} failed: {e}") from e

            if resp.status_code == 200:
                try:
                    return resp.json()
                except ValueError as e:
                    raise SystemOneError(
                        f"Could not decode JSON response from {self.endpoint}: {e}"
                    ) from e

            # Non-200: decide whether to retry.
            detail = self._error_detail(resp)
            if resp.status_code in _NO_RETRY_STATUSES or attempt >= self.max_retries:
                raise SystemOneError(
                    f"System One request failed [{resp.status_code}]: {detail}",
                    status_code=resp.status_code,
                )
            if resp.status_code in _RETRY_STATUSES:
                self._sleep_backoff(attempt)
                continue
            # Unknown status: do not retry.
            raise SystemOneError(
                f"System One request failed [{resp.status_code}]: {detail}",
                status_code=resp.status_code,
            )

        # Should not reach here; raise last captured network error if any.
        if last_exc is not None:
            raise SystemOneError(str(last_exc)) from last_exc
        raise SystemOneError("System One request failed after retries.")

    def _sleep_backoff(self, attempt: int):
        """Exponential backoff (1s, 2s, 4s, ...) with jitter."""
        delay = (2 ** attempt) + random.uniform(0, 1)
        time.sleep(delay)

    @staticmethod
    def _error_detail(resp) -> str:
        try:
            body = resp.json()
            if isinstance(body, dict):
                return body.get("error") or body.get("message") or json.dumps(body)
            return str(body)
        except Exception:
            return resp.text or "<no response body>"

    # ---------------------------------------------------------------- core

    @staticmethod
    def _build_questions(
        nouls: Optional[Sequence[str]],
        choices: Optional[Sequence[Tuple[str, Sequence[str]]]],
        scores: Optional[Sequence[Tuple[str, Sequence[str]]]],
        protocol: str = "systemone",
    ):
        """Build the `questions` payload from convenience arguments.

        For `protocol="systemone"` returns a map keyed `noul_0`, `choice_0`,
        `score_0`, ... For `protocol="decisions"` returns a list of question
        objects whose `name` fields use those same keys, mapping
        noul->predicate, choice criteria->choices list, score levels->levels list.
        """
        # Validate and normalize option/level lists once (shared by both protocols).
        norm_choices = []
        for q, options in (choices or []):
            opts = [o.strip() for o in options if o and o.strip()]
            if not opts:
                raise ValueError(f"choice {q!r} needs at least one option")
            norm_choices.append((q, opts))

        norm_scores = []
        for q, levels in (scores or []):
            lvls = [s.strip() for s in levels if s and s.strip()]
            if len(lvls) < 2:
                raise ValueError(f"score {q!r} needs at least two ordered levels")
            norm_scores.append((q, lvls))

        if protocol == "decisions":
            questions: List[Dict[str, Any]] = []
            for i, q in enumerate(nouls or []):
                # noul -> predicate (returns a probability in [0, 1]).
                questions.append({"type": "predicate", "name": f"noul_{i}",
                                   "instructions": q})
            for i, (q, opts) in enumerate(norm_choices):
                questions.append({
                    "type": "choice", "name": f"choice_{i}", "instructions": q,
                    "choices": [{"value": o, "description": ""} for o in opts],
                })
            for i, (q, lvls) in enumerate(norm_scores):
                questions.append({
                    "type": "score", "name": f"score_{i}", "instructions": q,
                    "levels": [{"label": s, "description": ""} for s in lvls],
                })
            return questions

        # Default: System One map.
        q_map: Dict[str, Dict[str, Any]] = {}
        for i, q in enumerate(nouls or []):
            q_map[f"noul_{i}"] = {"type": "noul", "instructions": q}
        for i, (q, opts) in enumerate(norm_choices):
            q_map[f"choice_{i}"] = {
                "type": "choice", "instructions": q,
                "criteria": {o: "" for o in opts},
            }
        for i, (q, lvls) in enumerate(norm_scores):
            q_map[f"score_{i}"] = {
                "type": "score", "instructions": q, "criteria": lvls,
            }
        return q_map

    def ask(
        self,
        state: Union[str, dict, list], # The content/context to evaluate (string, or object/array serialized per the API).
        *,
        nouls: Optional[Sequence[str]] = None, # Yes/no question instructions (each a string). Answered as a probability in [0, 1].
        choices: Optional[Sequence[Tuple[str, Sequence[str]]]] = None, # `(question, [option, ...])` pairs (pick one option).
        scores: Optional[Sequence[Tuple[str, Sequence[str]]]] = None, # `(question, [level, ...])` pairs; levels in ascending order (2-10 levels).
        questions: Optional[Dict[str, Dict[str, Any]]] = None, # Advanced: a raw `questions` map (per the API spec). Mutually exclusive with `nouls`/`choices`/`scores`.
    ) -> SystemOneResponse:
        """
        Ask one or more typed questions about `state` and return a
        `SystemOneResponse`.

        Provide questions either via the convenience arguments
        (`nouls`/`choices`/`scores`, auto-keyed `noul_0`, `choice_0`, ...) or via
        a raw `questions` map for full control over keys and criteria
        descriptions.

        **Returns:**

        - A `SystemOneResponse` whose `.answers[key]` exposes `.choice`,
          `.noul`, `.score`, `.probabilities`, and `.confidence`.
        """
        if questions is not None:
            if nouls or choices or scores:
                raise ValueError(
                    "`questions` is mutually exclusive with `nouls`/`choices`/`scores`."
                )
            # Advanced raw questions are assumed to already match the active
            # protocol's shape (map for systemone, list for decisions).
            q_payload = questions
        else:
            q_payload = self._build_questions(nouls, choices, scores,
                                              protocol=self.protocol)

        if not q_payload:
            raise ValueError("provide at least one noul, choice, or score question")

        if self.protocol == "decisions":
            payload = {"model": self.model, "input": state, "questions": q_payload}
        else:
            payload = {"model": self.model, "state": state, "questions": q_payload}
        return SystemOneResponse(self._post(payload), protocol=self.protocol)

    # ---------------------------------------------------------------- convenience

    def noul(self, state: Union[str, dict, list], question: str) -> float:
        """
        Ask a single yes/no question and return its probability (0.0 - 1.0).

        Example:

            d.noul("I was charged twice. Please refund.",
                   "Does the customer ask for a refund?")
            # 0.98
        """
        resp = self.ask(state, nouls=[question])
        return resp.answers["noul_0"].noul

    def choice(
        self, state: Union[str, dict, list], question: str, options: Sequence[str]
    ) -> Dict[str, float]:
        """
        Ask a single multiple-choice question and return a dict of
        `{option: probability}`.

        Example:

            d.choice("I was charged twice. Please refund.",
                     "Which team should handle this?",
                     ["billing", "technical", "other"])
            # {'billing': 0.98, 'technical': 0.01, 'other': 0.01}
        """
        resp = self.ask(state, choices=[(question, options)])
        return dict(resp.answers["choice_0"].probabilities or {})

    def score(
        self, state: Union[str, dict, list], question: str, levels: Sequence[str]
    ) -> float:
        """
        Ask a single ordinal-score question (levels ascending) and return the
        continuous score.

        Example:

            d.score("I was charged twice. Please refund.",
                    "How urgent is this ticket?",
                    ["Routine", "Soon", "Urgent"])
            # 1.40  (between "Soon" (1) and "Urgent" (2))
        """
        resp = self.ask(state, scores=[(question, levels)])
        return resp.answers["score_0"].score

    def classify(
        self,
        X: Union[str, List[str]], # A single text or a list of texts.
        labels: Sequence[str], # Candidate labels (the choice options).
        question: str = "Which category best describes the following content?", # The classification prompt.
        return_proba: bool = False, # If True, return `{label: probability}` dict(s) instead of the single top label.
    ):
        """
        Zero-shot classification: map each text in `X` to one of `labels`.

        A thin convenience wrapper over a single `choice` question. A list is
        returned when `X` is a list; a scalar when `X` is a string.

        Example:

            d.classify("I was charged twice. Please refund.",
                       ["billing", "technical", "other"])
            # 'billing'

            d.classify(["I was charged twice.", "The app crashes on login."],
                       ["billing", "technical"])
            # ['billing', 'technical']

            d.classify("I was charged twice.", ["billing", "technical"],
                       return_proba=True)
            # {'billing': 0.95, 'technical': 0.05}
        """
        single = isinstance(X, str)
        texts = [X] if single else list(X)

        results: List[Any] = []
        for text in texts:
            probs = self.choice(text, question, labels)
            if return_proba:
                results.append(probs)
            else:
                results.append(max(probs, key=probs.get) if probs else None)

        return results[0] if single else results


In [ ]:
show_doc(Decider.ask)


---

[source](https://github.com/amaiya/onprem/blob/master/onprem/pipelines/decider.py#L231){target="_blank" style="float:right; font-size:smaller"}

### Decider.ask

```python
def ask(
    state:Union[str, dict, list], # The content/context to evaluate (string, or object/array serialized per the API).
    *,
    nouls:Optional[Sequence[str]]=None, # Yes/no question instructions (each a string). Answered as a probability in [0, 1].
    choices:Optional[Sequence[Tuple[str, Sequence[str]]]]=None, # `(question, [option, ...])` pairs (pick one option).
    scores:Optional[Sequence[Tuple[str, Sequence[str]]]]=None, # `(question, [level, ...])` pairs; levels in ascending order (2-10 levels).
    questions:Optional[Dict[str, Dict[str, Any]]]=None, # Advanced: a raw `questions` map (per the API spec). Mutually exclusive with `nouls`/`choices`/`scores`.
)->__main__.SystemOneResponse:
```

*Ask one or more typed questions about `state` and return a*
`SystemOneResponse`.

Provide questions either via the convenience arguments
(`nouls`/`choices`/`scores`, auto-keyed `noul_0`, `choice_0`, ...) or via
a raw `questions` map for full control over keys and criteria
descriptions.

**Returns:**

- A `SystemOneResponse` whose `.answers[key]` exposes `.choice`,
  `.noul`, `.score`, `.probabilities`, and `.confidence`.

In [ ]:
show_doc(Decider.noul)


---

[source](https://github.com/amaiya/onprem/blob/master/onprem/pipelines/decider.py#L271){target="_blank" style="float:right; font-size:smaller"}

### Decider.noul

```python
def noul(
    state:Union[str, dict, list], question:str
)->float:
```

*Ask a single yes/no question and return its probability (0.0 - 1.0).*

Example:

    d.noul("I was charged twice. Please refund.",
           "Does the customer ask for a refund?")
    # 0.98

In [ ]:
show_doc(Decider.choice)


---

[source](https://github.com/amaiya/onprem/blob/master/onprem/pipelines/decider.py#L284){target="_blank" style="float:right; font-size:smaller"}

### Decider.choice

```python
def choice(
    state:Union[str, dict, list], question:str, options:Sequence[str]
)->Dict[str, float]:
```

*Ask a single multiple-choice question and return a dict of*
`{option: probability}`.

Example:

    d.choice("I was charged twice. Please refund.",
             "Which team should handle this?",
             ["billing", "technical", "other"])
    # {'billing': 0.98, 'technical': 0.01, 'other': 0.01}

In [ ]:
show_doc(Decider.score)


---

[source](https://github.com/amaiya/onprem/blob/master/onprem/pipelines/decider.py#L301){target="_blank" style="float:right; font-size:smaller"}

### Decider.score

```python
def score(
    state:Union[str, dict, list], question:str, levels:Sequence[str]
)->float:
```

*Ask a single ordinal-score question (levels ascending) and return the*
continuous score.

Example:

    d.score("I was charged twice. Please refund.",
            "How urgent is this ticket?",
            ["Routine", "Soon", "Urgent"])
    # 1.40  (between "Soon" (1) and "Urgent" (2))

In [ ]:
show_doc(Decider.classify)


---

[source](https://github.com/amaiya/onprem/blob/master/onprem/pipelines/decider.py#L318){target="_blank" style="float:right; font-size:smaller"}

### Decider.classify

```python
def classify(
    X:Union[str, List[str]], # A single text or a list of texts.
    labels:Sequence[str], # Candidate labels (the choice options).
    question:str='Which category best describes the following content?', # The classification prompt.
    return_proba:bool=False, # If True, return `{label: probability}` dict(s) instead of the single top label.
):
```

*Zero-shot classification: map each text in `X` to one of `labels`.*

A thin convenience wrapper over a single `choice` question. A list is
returned when `X` is a list; a scalar when `X` is a string.

Example:

    d.classify("I was charged twice. Please refund.",
               ["billing", "technical", "other"])
    # 'billing'

    d.classify(["I was charged twice.", "The app crashes on login."],
               ["billing", "technical"])
    # ['billing', 'technical']

    d.classify("I was charged twice.", ["billing", "technical"],
               return_proba=True)
    # {'billing': 0.95, 'technical': 0.05}

In [ ]:
#| hide
import nbdev

nbdev.nbdev_export()
